# IMPORTING ALL LIBRARIES

In [16]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder,StandardScaler,OneHotEncoder
import pickle

In [17]:
#loading the data
data=pd.read_csv('../data/Churn_Modelling.csv')

In [18]:
#First view of data
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [19]:
data=data.drop(columns=['RowNumber','CustomerId','Surname'])

In [20]:
#label encoding 
le_gen=LabelEncoder()
data['Gender']=le_gen.fit_transform(data['Gender'])

In [21]:
#Onehot
onehot_geo=OneHotEncoder(handle_unknown='ignore')
geo_encoded=onehot_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df=pd.DataFrame(geo_encoded,columns=onehot_geo.get_feature_names_out(['Geography']))
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [22]:
data=pd.concat([data.drop('Geography',axis=1),geo_encoded_df],axis=1)

In [23]:
x=data.drop(['EstimatedSalary'],axis=1)

In [24]:
y=data['EstimatedSalary']

In [25]:
x.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,0,0.0,0.0,1.0


#### Train test split

In [29]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.20,random_state=42)

In [30]:
#scaling the datan
scaler=StandardScaler()
x_train=scaler.fit_transform(x_train)
x_test=scaler.transform(x_test)

In [31]:
#saving the processed encoder and scaled data
with open('le_gen.pkl','wb') as file:
    pickle.dump(le_gen,file)

with open('onehot_geo.pkl','wb') as file:
    pickle.dump(onehot_geo,file)

with open('scaler.pkl','wb') as file:
    pickle.dump(scaler,file)

# ANN Regression Problem statement

In [32]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [37]:
model=Sequential([
    Dense(64,activation='relu',input_shape=(x_train.shape[1],)),
    Dense(32,activation='relu'),
    Dense(1) #linear activation function
])

#compile the model
model.compile(optimizer='adam',loss='mean_absolute_error',metrics=['mae'])
model.summary()



c:\Users\himan\miniconda3\envs\annenv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [38]:
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime
log_dir='regressionlogs/fit'+datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback=TensorBoard(log_dir=log_dir,histogram_freq=1)

In [39]:
early_stoping_callback=EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)

In [40]:
historyy=model.fit(
    x_train,y_train,
    validation_data=(x_test,y_test),
    epochs=100,
    callbacks=[tensorboard_callback,early_stoping_callback]
    )

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - loss: 100386.2500 - mae: 100386.2500 - val_loss: 98554.8906 - val_mae: 98554.8906
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 99800.5625 - mae: 99800.5625 - val_loss: 97380.8906 - val_mae: 97380.8906
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 97770.2266 - mae: 97770.2266 - val_loss: 94392.0859 - val_mae: 94392.0859
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 93732.9922 - mae: 93732.9922 - val_loss: 89281.2344 - val_mae: 89281.2344
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 87692.7500 - mae: 87692.7500 - val_loss: 82456.6016 - val_mae: 82456.6016
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 80118.4453 - mae: 80118.4453 - val_loss: 74613.6875 - val_mae: 74613.6875
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 72011.1719 - mae: 72011.1719 - val_loss: 67043.6016 - val_mae: 67043.6016
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step 

In [41]:
%load_ext tensorboard

In [43]:
#Tensoboard setup (web server)
import subprocess

process = subprocess.Popen([
    "python",
    "-m",
    "tensorboard.main",
    "--logdir",
    r"D:\Ann project\model\regressionlogs\fit20260921-133151",
    "--port",
    "6008"
])

In [44]:
#saving the trained model
model.save('model_reg.keras')